# B1 · What AlphaFold's convolutions actually do, and what replaced them
**Series B: Deep Learning for Molecules, Post 1 of 7** &nbsp;|&nbsp; companion *primer* notebook

The lecture notes introduce **convolutional neural networks** through AlphaFold: AlphaFold1 used a CNN to turn a protein into a 2D map of distances between residues, and AlphaFold2, the model of the ColabFold notebook that this post is built on, replaced that CNN by an attention transformer. This primer works with **the structure that the notebook predicted** (59 residues, the ColabFold default sequence; its C$\alpha$ coordinates and confidence are embedded below, nothing to download): it builds a convolution **from scratch**, shows **what a convolution sees** in the real distance map, turns a distance map back into coordinates (the last step of AlphaFold1's recipe) and finds out what a distance map *cannot* tell, compares convolution with attention, and reads the notebook's own outputs.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in a few seconds on a CPU. **Part II** applies the same analysis to **your own ColabFold run**, using the result files that Notebook 1 produces.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | a convolution from scratch: output size, parameters, equivariance, dilation |
| 2 | the notebook's structure as an image: the distance map and the contact map |
| 3 | what convolutions see in that image |
| 4 | from distances to coordinates, and the mirror image |
| 5 | convolution against attention |
| 6 | skip connections and the vanishing gradient |
| 7 | reading the AlphaFold2 log of the notebook |
| **Part II** | the same analysis of your own ColabFold result |

## 0. Setup and data

In [ ]:
import math, time, glob, json, os, zipfile, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
SEQ = "PIAQIHILEGRSDEQKETLIREVSEAISRSLDAPLTSVRVIITEMAKGHFGIGGELASK"     # the notebook's query sequence (59 residues)

In [ ]:
CA_PLDDT = """
x y z plddt
11.098 3.405 -6.924 97.31
9.332 4.519 -3.877 98.44
5.568 4.010 -3.318 98.50
3.710 5.798 -0.539 98.62
0.360 4.370 0.297 98.19
-2.086 6.000 2.645 97.50
-4.827 3.796 4.077 97.25
-7.285 3.896 6.936 96.75
-6.238 1.957 9.996 96.62
-7.748 -1.461 10.689 96.69
-6.018 -3.725 8.253 97.81
-4.335 -6.882 9.506 98.44
-0.561 -7.379 9.264 98.31
-1.221 -10.077 6.664 98.62
-3.128 -7.685 4.455 98.62
-0.410 -5.039 4.745 98.69
2.182 -7.668 3.822 98.62
0.145 -8.717 0.862 98.75
-0.239 -5.076 -0.234 98.69
3.518 -4.399 0.071 98.75
4.311 -7.530 -1.986 98.75
1.904 -6.855 -4.694 98.75
2.760 -3.181 -5.161 98.62
6.499 -3.959 -5.148 98.81
5.980 -6.639 -7.873 98.81
3.808 -4.347 -9.921 98.75
6.358 -1.568 -9.851 98.75
9.246 -3.881 -10.538 98.69
7.407 -5.401 -13.554 98.56
6.176 -2.121 -14.973 98.62
9.376 -0.241 -14.759 98.38
11.809 -3.122 -15.204 98.19
13.447 -2.221 -11.848 98.44
15.066 -4.774 -9.543 98.50
12.641 -5.784 -6.831 98.44
15.309 -5.108 -4.270 98.38
15.350 -1.447 -5.241 98.44
11.534 -0.962 -4.576 98.69
10.472 0.596 -1.263 98.69
6.793 0.633 -0.127 98.69
5.751 2.748 2.793 98.69
2.308 2.319 4.352 98.56
0.947 5.231 6.349 98.12
-2.147 4.472 8.355 97.62
-4.718 7.157 9.190 96.81
-6.962 6.997 12.127 96.88
-10.623 7.351 10.961 94.88
-10.913 10.663 12.808 96.12
-7.974 12.135 10.906 95.38
-9.325 11.399 7.458 95.56
-12.150 13.311 5.661 94.81
-14.122 12.017 2.724 94.31
-16.419 14.581 1.057 94.69
-16.248 16.933 4.082 95.88
-17.065 14.157 6.688 96.94
-14.806 12.278 8.983 95.62
-14.110 8.783 7.842 93.81
-15.144 7.770 11.283 93.19
-18.660 9.056 10.585 84.31
"""

In [ ]:
arr = np.array([[float(v) for v in line.split()] for line in CA_PLDDT.strip().splitlines()[1:]]); xyz, plddt = arr[:, :3], arr[:, 3]; N = len(xyz)
print(f"{N} residues | C-alpha coordinates {xyz.shape} (angstrom) | pLDDT: mean {plddt.mean():.1f}, min {plddt.min():.1f} (residue {int(plddt.argmin()) + 1}), max {plddt.max():.1f}")
step = np.linalg.norm(np.diff(xyz, axis=0), axis=1); print(f"distance between successive C-alpha atoms: mean {step.mean():.2f} A, sd {step.std():.3f} A (a peptide bond fixes it near 3.8 A)")
print(f"mean pLDDT of residues 1 to 45: {plddt[:45].mean():.1f} | of the last ten residues: {plddt[-10:].mean():.1f} | the only residue below 90: {[int(i) + 1 for i in np.where(plddt < 90)[0]]}")
assert N == len(SEQ) == 59 and abs(step.mean() - 3.8) < 0.05 and plddt[:45].mean() > 98 and plddt[-10:].mean() < 95

## 1. A convolution from scratch
A kernel $\omega$ of size $k_h\times k_w$ slides over the input $z$; at each position the overlapping values are multiplied by the kernel weights and summed, $z'_{ij}=\varphi\big(\sum_{k,l}z_{i+d\,k,\,j+d\,l}\,\omega_{kl}\big)$, with a **stride** $s$ (the step), a **padding** $p$ (zeros around the border) and a **dilation** $d$ (the spacing of the kernel elements). The output has $W_{out}=\big\lfloor\frac{W_{in}+2p-d(k_w-1)-1}{s}\big\rfloor+1$ columns (and likewise rows). One kernel is shared by all positions, which is what makes a convolution cheap.
**Stop and predict:** for a $7\times7$ input and a $3\times3$ kernel (stride 1, no padding), the size of the output and the number of weights, against a dense layer with the same input and output sizes.

In [ ]:
def conv2d(x, k, stride=1, pad=0, dil=1):
    """Cross-correlation of a 2D array x with a kernel k (the 'convolution' of deep learning): written with loops, to see what it does."""
    kh, kw = k.shape; xp = np.pad(x, pad); H = (xp.shape[0] - dil * (kh - 1) - 1) // stride + 1; W = (xp.shape[1] - dil * (kw - 1) - 1) // stride + 1
    out = np.zeros((H, W))
    for a in range(H):
        for b in range(W): out[a, b] = sum(xp[a * stride + i * dil, b * stride + j * dil] * k[i, j] for i in range(kh) for j in range(kw))
    return out
out_size = lambda n, k, s=1, p=0, d=1: (n + 2 * p - d * (k - 1) - 1) // s + 1
rng = np.random.default_rng(0); bad = 0; tested = 0
for n in (7, 10, 12):
    for k in (2, 3, 5):
        for s in (1, 2, 3):
            for p in (0, 1, 2):
                for d in (1, 2, 3):
                    if out_size(n, k, s, p, d) < 1: continue
                    tested += 1; bad += conv2d(rng.normal(size=(n, n)), rng.normal(size=(k, k)), s, p, d).shape != (out_size(n, k, s, p, d),) * 2
print(f"the output-size formula agrees with the loop implementation in {tested - bad} of {tested} settings of size, kernel, stride, padding and dilation")
x7 = rng.normal(size=(7, 7)); y5 = conv2d(x7, rng.normal(size=(3, 3))); print("7 x 7 input, 3 x 3 kernel ->", y5.shape, "| weights: convolution", 3 * 3, "against a dense layer", 7 * 7 * 5 * 5)
assert bad == 0 and y5.shape == (5, 5)

In [ ]:
# the kernel of the lecture notes compares the two diagonal corners of a 3 x 3 window: silent on flat regions and on vertical edges, active along a diagonal step
edge = np.array([[1, 0, -1], [0, 0, 0], [-1, 0, 1]], float); step_img = np.tril(np.ones((8, 8))); resp = conv2d(step_img, edge); nz = np.argwhere(resp != 0)
vert = np.zeros((8, 8)); vert[:, 4:] = 1
print("edge kernel: flat image ->", int((conv2d(np.ones((8, 8)), edge) != 0).sum()), "nonzero responses | vertical edge ->", int((conv2d(vert, edge) != 0).sum()), f"| diagonal step -> {len(nz)} nonzero responses, all within |row - col| <= {int(np.abs(nz[:, 0] - nz[:, 1]).max())} of the diagonal, values {np.unique(resp)}")
assert not conv2d(np.ones((8, 8)), edge).any() and not conv2d(vert, edge).any() and np.abs(nz[:, 0] - nz[:, 1]).max() <= 2
# a convolution commutes with translation: shift the input, and the output is shifted by the same amount
big = np.zeros((20, 20)); big[4:9, 5:10] = rng.normal(size=(5, 5)); ker = rng.normal(size=(3, 3))
shifted = np.roll(big, (3, 4), axis=(0, 1)); a_, b_ = conv2d(big, ker), conv2d(shifted, ker)
print("convolution of the shifted input equals the shifted convolution (shift 3 rows, 4 columns):", np.allclose(np.roll(a_, (3, 4), axis=(0, 1)), b_))
# parameters of a layer with channels and the receptive field of stacked dilated convolutions
params = lambda cin, cout, kh, kw: cout * (cin * kh * kw + 1)
print("a layer with 96 kernels of 11 x 11 on 3 channels (the first layer of AlexNet):", params(3, 96, 11, 11), "parameters")
rf = lambda dils, k=3: 1 + (k - 1) * sum(dils)
print("receptive field of 3 x 3 layers: plain x15:", rf([1] * 15), "| dilations 1, 2, 4, 8:", rf([1, 2, 4, 8]), "| dilations 1, 2, 4, 8 twice:", rf([1, 2, 4, 8] * 2))
assert np.allclose(np.roll(a_, (3, 4), axis=(0, 1)), b_) and params(3, 96, 11, 11) == 34944 and rf([1, 2, 4, 8]) == rf([1] * 15) == 31
print("four dilated layers (1, 2, 4, 8) see as far as fifteen plain 3 x 3 layers")

## 2. The structure as an image
The **distance map** of a protein is the $N\times N$ matrix of distances between the C$\alpha$ atoms of residues $j$ and $k$, $d_{jk}=\|x_j-x_k\|$: a picture whose pixels are residue pairs. It is symmetric, its diagonal is $0$, and it does not depend on how the molecule is placed or rotated. A **contact map** marks the pairs closer than 8 \AA, excluding sequence neighbors ($|j-k|<3$). Local structure (helices, turns) lives near the diagonal; pairs far apart in sequence but close in space lie far from it.
**Stop and predict:** how many contacts will a 59-residue protein have, and what share will be between residues far apart in the sequence?

In [ ]:
D = np.linalg.norm(xyz[:, None] - xyz[None], axis=-1); I, J = np.indices(D.shape); sep = np.abs(I - J); C = (D < 8) & (sep >= 3)
print(f"distance map {D.shape}: symmetric {np.allclose(D, D.T)}, largest distance {D.max():.1f} A")
n_c = C.sum() // 2; loc = (C & (sep <= 5)).sum() // 2; med = (C & (sep >= 6) & (sep <= 11)).sum() // 2; lng = (C & (sep >= 12)).sum() // 2
print(f"contacts (< 8 A, |j - k| >= 3): {n_c} pairs | local (|j - k| 3 to 5): {loc} | medium (6 to 11): {med} | long-range (>= 12): {lng} ({100 * lng / n_c:.0f} %)")
assert n_c == loc + med + lng == 127 and lng == 46

In [ ]:
def torsion(p0, p1, p2, p3):
    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2; b1 = b1 / np.linalg.norm(b1); v = b0 - b0 @ b1 * b1; w = b2 - b2 @ b1 * b1
    return float(np.degrees(np.arctan2(np.cross(b1, v) @ w, v @ w)))
def helix_windows(X):
    Dx = np.linalg.norm(X[:, None] - X[None], axis=-1); return [i for i in range(len(X) - 4) if 4.6 < Dx[i, i + 3] < 5.8 and 5.6 < Dx[i, i + 4] < 6.9]    # alpha-helix: d(i,i+3) ~ 5.0-5.4 A, d(i,i+4) ~ 6.2 A
def crude_ss(X):
    """a crude secondary-structure string from C-alpha distances only: H helix, E extended (d(i,i+3) > 8.5 A), - other (not DSSP)"""
    Dx = np.linalg.norm(X[:, None] - X[None], axis=-1); n = len(X); h = np.zeros(n, bool); e = np.zeros(n, bool)
    for i in range(n - 4):
        if 4.6 < Dx[i, i + 3] < 5.8 and 5.6 < Dx[i, i + 4] < 6.9: h[i:i + 5] = True
    for i in range(n - 3):
        if Dx[i, i + 3] > 8.5: e[i:i + 4] = True
    e &= ~h; return "".join("H" if h[i] else ("E" if e[i] else "-") for i in range(n))
ss = crude_ss(xyz); hw = helix_windows(xyz); tors = [torsion(*xyz[i:i + 4]) for i in hw]
print(SEQ); print(ss); print(f"helix {ss.count('H')}, extended {ss.count('E')}, other {ss.count('-')} residues | {len(hw)} helical windows with a mean C-alpha pseudo-torsion of {np.mean(tors):+.1f} degrees (a right-handed alpha helix: about +50)")
assert ss.count("H") >= 20 and ss.count("E") >= 20 and np.mean(tors) > 30

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.9)); im0 = ax[0].imshow(D, cmap="viridis_r"); plt.colorbar(im0, ax=ax[0], fraction=0.046, label="distance (angstrom)")
ax[0].set(title="(a) distance map", xlabel="residue", ylabel="residue"); ax[1].imshow(C, cmap="Greys"); ax[1].set(title="(b) contact map (< 8 A, |j - k| >= 3)", xlabel="residue", ylabel="residue")
ax[2].plot(np.arange(1, N + 1), plddt, "o-", ms=3, color=COLORS["blue"]); ax[2].axhline(90, color=COLORS["gray"], ls=":"); ax[2].set(title="(c) confidence of the model (pLDDT)", xlabel="residue", ylabel="pLDDT", ylim=(80, 100))
plt.tight_layout(); plt.show()

## 3. What convolutions see in the map
In a contact map, structure shows up as **lines**: a helix is a band parallel to the main diagonal; two strands that run side by side in the *same* direction (a parallel $\beta$ pair) make a line parallel to the diagonal, away from it; two strands in *opposite* directions (an antiparallel pair, a hairpin) make a line perpendicular to the diagonal. A convolution is a local pattern detector: a kernel with ones along a diagonal responds where five contacts in a row run along a diagonal, and the flipped kernel where they run the other way.
**Stop and predict:** which of the two kernels, if any, will find lines in this protein?

In [ ]:
Cf = C.astype(float); diag, anti = np.eye(5), np.fliplr(np.eye(5)); Rd, Ra = conv2d(Cf, diag), conv2d(Cf, anti)
pos_d = [(a + 3, b + 3) for a, b in np.argwhere(Rd >= 5) if b > a]; pos_a = [(a + 3, b + 3) for a, b in np.argwhere(Ra >= 5) if b > a]            # residue numbers of the centres, upper triangle
print(f"contact map {Cf.shape} -> response {Rd.shape} (59 - 5 + 1 = 55)")
print(f"five contacts in a row along a diagonal: {len(pos_d)} positions in the upper triangle | along an anti-diagonal: {len(pos_a)}")
far = [(i, j) for i, j in pos_d if j - i >= 12]; print("far from the main diagonal (|j - i| >= 12):", len(far), "positions; residues i from", min(i for i, j in far), "to", max(i for i, j in far), "paired with j from", min(j for i, j in far), "to", max(j for i, j in far))
assert len(pos_a) == 0 and len(far) > 0

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5)); ax.imshow(Cf, cmap="Greys", alpha=0.45)
ax.scatter([j - 1 for i, j in pos_d], [i - 1 for i, j in pos_d], s=10, color=COLORS["red"], label="diagonal line (5 contacts)"); ax.scatter([i - 1 for i, j in pos_d], [j - 1 for i, j in pos_d], s=10, color=COLORS["red"])
ax.set(xlabel="residue", ylabel="residue", title="Where the diagonal kernel fires"); ax.legend(fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()

## 4. From distances to coordinates, and the mirror image
AlphaFold1's second step turns predicted distances into a structure by optimization. Here is the cleanest version of that step. Classical **multidimensional scaling** (MDS) builds coordinates from a complete distance matrix in one step: double-center the squared distances, $B=-\tfrac12JD^{\circ2}J$, and take the three largest eigenvectors, scaled by the square roots of their eigenvalues. For distances that come from three-dimensional points, $B$ has exactly three non-zero eigenvalues.
**Stop and predict:** can a distance map distinguish a protein from its mirror image?

In [ ]:
def kabsch(P, Q, allow_reflection=False):
    """RMSD after the best rotation (and, optionally, a reflection) of P onto Q"""
    P0, Q0 = P - P.mean(0), Q - Q.mean(0); U, S, Vt = np.linalg.svd(P0.T @ Q0); d = 1.0 if allow_reflection else np.sign(np.linalg.det(U @ Vt)); R = U @ np.diag([1, 1, d]) @ Vt
    return float(np.sqrt(((P0 @ R - Q0) ** 2).sum(1).mean()))
def mds(Dm):
    n = len(Dm); J = np.eye(n) - 1.0 / n; w, V = np.linalg.eigh(-0.5 * J @ (Dm ** 2) @ J); idx = np.argsort(w)[::-1][:3]; return V[:, idx] * np.sqrt(np.maximum(w[idx], 0)), np.sort(w)[::-1]
X, eig = mds(D); mirror = X * np.array([1, 1, -1])
print("eigenvalues of the double-centred matrix, first 5:", eig[:5].round(1), "-> exactly three are non-zero")
print(f"MDS from the exact map: RMSD to the notebook's structure {kabsch(X, xyz):.3f} A | helical pseudo-torsion {np.mean([torsion(*X[i:i + 4]) for i in hw]):+.1f} degrees")
print(f"the mirror image: same distance map: {np.allclose(np.linalg.norm(mirror[:, None] - mirror[None], axis=-1), D, atol=1e-6)} | RMSD to the structure {kabsch(mirror, xyz):.3f} A without a reflection, {kabsch(mirror, xyz, True):.3f} A with one | helical pseudo-torsion {np.mean([torsion(*mirror[i:i + 4]) for i in hw]):+.1f} degrees")
assert (np.abs(eig[3:]) < 1e-6).all() and kabsch(X, xyz, True) < 1e-6 and kabsch(mirror, xyz) > 3 and np.allclose(np.linalg.norm(mirror[:, None] - mirror[None], axis=-1), D, atol=1e-6)

In [ ]:
def stress_refine(X0, Dm, steps=400, lr=0.05):
    X_ = X0.copy(); n = len(X_)
    for _ in range(steps):
        diff = X_[:, None] - X_[None]; dist = np.linalg.norm(diff, axis=-1) + np.eye(n); g = ((dist - Dm) / dist)[:, :, None] * diff; X_ -= lr * g.sum(1) / (n - 1)
    return X_
rng = np.random.default_rng(0); rows = []
for sig in (0.0, 0.5, 1.0, 2.0, 3.0):
    E_ = np.triu(rng.normal(0, sig, (N, N)), 1); Dn = np.maximum(D + E_ + E_.T, 0.1); np.fill_diagonal(Dn, 0); X0 = mds(Dn)[0]; X1 = stress_refine(X0, Dn)
    hand = np.sign(np.mean([torsion(*X1[i:i + 4]) for i in hw])); rows.append((sig, kabsch(X0, xyz, True), kabsch(X1, xyz, True), "right-handed" if hand > 0 else "MIRROR IMAGE"))
tab = pd.DataFrame(rows, columns=["noise on every distance (A)", "RMSD of MDS (A)", "RMSD after refinement (A)", "handedness of the result"]); print(tab.round(2).to_string(index=False))
assert tab.iloc[0, 2] < 1e-6 and tab.iloc[1, 2] < 0.5 and tab.iloc[3, 2] < 2.0 and (tab.iloc[:, 2].diff().dropna() > -1e-9).all()
print("\nRMSD is measured with a reflection allowed; a distance map cannot tell the two hands apart, so the hand of a result must be fixed from the geometry, for instance from the sign of the helical torsions")

## 5. Convolution against attention
AlphaFold2 replaced the convolutional trunk by **attention**. Both are linear maps of the input at each step, but different ones. A convolution is multiplication by a fixed **banded Toeplitz matrix**: row $t$ contains the kernel, shifted by $t$, whatever the input. Attention computes its matrix from the input itself, $A=\mathrm{softmax}(QK^T/\sqrt d)$, a full matrix whose rows give every position a weight on every other position.
**Stop and predict:** if the input changes, does the matrix of a convolution change? And that of attention?

In [ ]:
n = 8; kern = np.array([1.0, 2.0, 1.0]); x1 = rng.normal(size=n)
T = np.zeros((n, n)); 
for t in range(n):
    for j, w in enumerate(kern): 
        if 0 <= t + j - 1 < n: T[t, t + j - 1] = w                   # row t holds the kernel centred on t (zero padding)
print("convolution equals multiplication by a banded Toeplitz matrix:", np.allclose(np.convolve(x1, kern[::-1], mode="same"), T @ x1))
Wq, Wk = rng.normal(size=(n, 4)), rng.normal(size=(n, 4))
def attn_matrix(x):
    emb = np.outer(x, np.ones(n)) * 0 + x[:, None] * np.linspace(0.5, 1.5, n)[None, :]                   # a toy embedding that depends on the input
    q, k = emb @ Wq, emb @ Wk; s = q @ k.T / 2.0; e = np.exp(s - s.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
x2 = rng.normal(size=n); A1, A2 = attn_matrix(x1), attn_matrix(x2)
print("rows of the attention matrix sum to 1:", np.allclose(A1.sum(1), 1), "| the convolution matrix does not depend on the input (same T for any x);", f"the attention matrix does: largest change {np.abs(A1 - A2).max():.2f}")
print(f"nonzero entries per row: convolution {int((T != 0).sum(1).max())} of {n}, attention {int((A1 > 1e-3).sum(1).mean() + 0.5)} of {n} on average (every position can look at every other)")
assert np.allclose(np.convolve(x1, kern[::-1], mode="same"), T @ x1) and np.abs(A1 - A2).max() > 0.1

## 6. Skip connections and the vanishing gradient
The notes explain that **residual layers**, which skip over convolutional layers, made very deep networks trainable by avoiding the vanishing gradient (the subject of Post A3). We measure the effect: in stacks of width 32 with random weights, compare a plain layer $h\leftarrow\tanh(Wh)$ with a residual layer $h\leftarrow h+\tanh(Wh)$, and look at the size of the gradient of the output at the first layer relative to the last, for 10, 25 and 50 layers.
**Stop and predict:** which of the two will keep a usable gradient at the first layer?

In [ ]:
def grad_ratio(residual, depth=50, width=32, seed=0, scale=1.0):
    r = np.random.default_rng(seed); Ws = [r.normal(0, scale / np.sqrt(width), (width, width)) for _ in range(depth)]; h = r.normal(size=width); hs = [h]
    for W in Ws: z = W @ h; h = h + np.tanh(z) if residual else np.tanh(z); hs.append(h)
    g = np.ones(width); n0 = np.linalg.norm(g)
    for W, hprev in zip(Ws[::-1], hs[-2::-1]):
        local = 1 - np.tanh(W @ hprev) ** 2; g = (g + W.T @ (g * local)) if residual else W.T @ (g * local)
    return np.linalg.norm(g) / n0
rows = [(sc, dp_, np.median([grad_ratio(False, dp_, seed=k, scale=sc) for k in range(10)]), np.median([grad_ratio(True, dp_, seed=k, scale=sc) for k in range(10)])) for sc in (0.5, 0.8) for dp_ in (10, 25, 50)]
gt = pd.DataFrame(rows, columns=["weight scale", "layers", "plain: gradient ratio", "residual: gradient ratio"]); print("gradient at the first layer divided by the gradient at the last (median of 10 random initializations; weights with standard deviation scale / sqrt(width)):")
print(gt.to_string(index=False, formatters={"weight scale": "{:.1f}".format, "plain: gradient ratio": "{:.1e}".format, "residual: gradient ratio": "{:.1e}".format}))
plain, resid = gt.loc[(gt["weight scale"] == 0.5) & (gt.layers == 50), "plain: gradient ratio"].item(), gt.loc[(gt["weight scale"] == 0.5) & (gt.layers == 50), "residual: gradient ratio"].item()
print("\nthe plain gradient shrinks by a constant factor per layer when the weights are small and is gone after 50 layers; the residual layer adds the identity to the Jacobian, so the gradient does not vanish (here it even grows slowly with the depth)")
assert plain < 1e-10 and resid > 1e-2 and gt.iloc[0, 2] > gt.iloc[1, 2] > gt.iloc[2, 2]

## 7. Reading the AlphaFold2 log of the notebook
The notebook ran AlphaFold2 (`alphafold2_ptm` weights) with five models and up to three recycles on the 59-residue query. The log lines below are typed in from its saved output: the pLDDT and pTM after each recycle, and the times.
**Stop and predict:** does recycling improve the confidence for an easy target?

In [ ]:
log = {1: [(96.6, .753), (96.4, .757), (96.1, .753), (95.5, .750)], 2: [(96.9, .761), (97.0, .767), (96.7, .762), (96.5, .763)], 3: [(97.2, .775), (97.4, .783), (97.4, .782), (97.4, .782)],
       4: [(97.2, .772), (97.2, .778), (96.9, .776), (96.6, .775)], 5: [(97.0, .781), (96.4, .781), (96.2, .778), (96.2, .777)]}
took = {1: 44.5, 2: 13.0, 3: 13.1, 4: 13.2, 5: 13.5}
rows = [(m, v[0][0], v[-1][0], v[-1][0] - v[0][0], v[0][1], v[-1][1], took[m]) for m, v in log.items()]
tab = pd.DataFrame(rows, columns=["model", "pLDDT recycle 0", "pLDDT recycle 3", "change", "pTM recycle 0", "pTM recycle 3", "seconds"]); print(tab.round(3).to_string(index=False))
dp, dt = np.mean([r[3] for r in rows]), np.mean([r[5] - r[4] for r in rows]); print(f"\nmean change from recycle 0 to 3: pLDDT {dp:+.2f}, pTM {dt:+.3f} | the first model took {took[1]:.1f} s against about {np.mean([took[m] for m in range(2, 6)]):.1f} s for the others: about {took[1] - np.mean([took[m] for m in range(2, 6)]):.0f} s of one-time compilation")
final = sorted(((v[-1][0], m) for m, v in log.items()), reverse=True); print("ranking by pLDDT after the last recycle:", [(f"model {m}", p) for p, m in final], "| the rank-1 structure is the one embedded above")
assert abs(dp + 0.54) < 0.01 and final[0][1] == 3

## 8. Your turn
**Exercise 1.** Use a $3\times3$ kernel with stride 2 and padding 1 on the contact map. What is the output size, and what does the map look like?
**Exercise 2.** Add a diagonal kernel of size 7. Does it still find the lines far from the diagonal?
**Exercise 3.** Reconstruct the structure from only the pairs closer than 12 \AA. Why does a plain gradient descent from a random start often fail?
**Exercise 4.** Change the contact threshold to 6 \AA and 10 \AA. How do the numbers of contacts and of detections change?
**Exercise 5.** Make the residual layer $h\leftarrow h+0.1\tanh(Wh)$. How does the gradient ratio change with the depth, and at which weight scale does the plain stack stop vanishing?

## 9. Self-check

In [ ]:
assert bad == 0 and len(pos_a) == 0 and len(far) > 0 and n_c == 127 and kabsch(X, xyz, True) < 1e-6 and kabsch(mirror, xyz) > 3
assert plain < 1e-10 and resid > 1e-2 and abs(dp + 0.54) < 0.01
print("All checks passed: you reproduced the numbers in the post.")

# Part II. The same analysis of your own ColabFold run
> **Read this first.** Run **Notebook 1** (the ColabFold AlphaFold2 notebook) on any monomer sequence in the *same* Colab session; it creates a folder named after the job and a file `jobname.result.zip`. Part II looks for the best-ranked structure (`*_rank_001_*.pdb`) and its scores (`*_scores_rank_001_*.json`) in the current folder, its subfolders, or in a result zip, and repeats the analysis of Part I on it: the distance and contact maps, the crude secondary structure, the diagonal detectors, the exact three-dimensionality of the map, and, if the scores are found, the relation between the confidence and the predicted aligned error. You can also upload the zip (`files.upload()` in Colab) first. If nothing is found, the cell says so and nothing else runs.

In [ ]:
def find_results(root="."):
    zips = sorted(glob.glob(os.path.join(root, "*.result.zip")))
    pdbs = sorted(glob.glob(os.path.join(root, "**", "*_rank_001_*.pdb"), recursive=True))
    if not pdbs and zips:
        with zipfile.ZipFile(zips[-1]) as z: z.extractall(os.path.join(root, "colabfold_result_unzipped"))
        pdbs = sorted(glob.glob(os.path.join(root, "colabfold_result_unzipped", "**", "*_rank_001_*.pdb"), recursive=True))
    pdbs = [p for p in pdbs if "unrelaxed" in p] or pdbs                                # the unrelaxed structure is the model's own output
    if not pdbs: return None, None
    js = sorted(glob.glob(os.path.join(os.path.dirname(pdbs[0]), "*_scores_rank_001_*.json"))); return pdbs[0], (js[0] if js else None)
def read_ca(path):
    """C-alpha coordinates, residue names and the B-factor column (the pLDDT in AlphaFold files) of the first chain"""
    xs, names, bf, chain0 = [], [], [], None
    for line in open(path):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            ch = line[21]; chain0 = chain0 or ch
            if ch != chain0: continue
            xs.append([float(line[30:38]), float(line[38:46]), float(line[46:54])]); names.append(line[17:20]); bf.append(float(line[60:66]))
    return np.array(xs), names, np.array(bf)
pdb_path, json_path = find_results(); checks2 = []
def check2(label, ok): checks2.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if pdb_path is None: print("No ColabFold result found in the current folder: run Notebook 1 in this session (or upload its result zip) and re-run this cell.")
else: print("structure:", pdb_path, "| scores:", json_path)

In [ ]:
if pdb_path is not None:
    try:
        Xr, names_r, plddt_r = read_ca(pdb_path); Nr = len(Xr); Dr = np.linalg.norm(Xr[:, None] - Xr[None], axis=-1); Ir, Jr = np.indices(Dr.shape); sepr = np.abs(Ir - Jr); Cr = (Dr < 8) & (sepr >= 3)
        step_r = np.linalg.norm(np.diff(Xr, axis=0), axis=1); nc = Cr.sum() // 2; lg = (Cr & (sepr >= 12)).sum() // 2; ssr = crude_ss(Xr); hwr = helix_windows(Xr)
        Rr = conv2d(Cr.astype(float), np.eye(5)); Rar = conv2d(Cr.astype(float), np.fliplr(np.eye(5))); far_r = [(a, b) for a, b in np.argwhere(Rr >= 5) if b - a >= 12]
        rowsr = [("residues", Nr, N), ("mean pLDDT", round(float(plddt_r.mean()), 1), round(float(plddt.mean()), 1)), ("contacts (< 8 A, |j-k| >= 3)", int(nc), int(n_c)), ("long-range contacts (>= 12 apart)", int(lg), int(lng)),
                 ("helix residues (crude)", ssr.count("H"), ss.count("H")), ("extended residues (crude)", ssr.count("E"), ss.count("E")), ("diagonal-line detections (5 in a row)", int((Rr >= 5).sum()), int((Rd >= 5).sum())),
                 ("anti-diagonal-line detections", int((Rar >= 5).sum()), int((Ra >= 5).sum()))]
        print(pd.DataFrame(rowsr, columns=["quantity", "your run", "the notebook's example"]).to_string(index=False)); print("\nyour sequence:", "".join({"ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V"}.get(a, "X") for a in names_r)); print("crude secondary structure:", ssr)
        check2(f"the distance between successive C-alpha atoms is {step_r.mean():.2f} A (a peptide bond fixes it near 3.8 A)", abs(step_r.mean() - 3.8) < 0.1)
        eig_r = mds(Dr)[1]; check2(f"the distance map of your model is exactly three-dimensional: the 4th eigenvalue is {eig_r[3]:.1e} against {eig_r[0]:.0f} for the 1st", abs(eig_r[3]) < 1e-3 * eig_r[0])
        Xm_r = mds(Dr)[0]; check2("MDS rebuilds your model from its distance map (RMSD, reflection allowed, below 0.01 A)", kabsch(Xm_r, Xr, True) < 0.01)
    except Exception as e: print("analysis could not be completed:", type(e).__name__, e)

In [ ]:
if pdb_path is not None and json_path is not None:
    try:
        sc = json.load(open(json_path)); pae = np.array(sc["pae"]) if "pae" in sc else None
        print("keys of the scores file:", list(sc.keys()))
        if pae is not None:
            print(f"predicted aligned error: matrix {pae.shape}, mean {pae.mean():.2f} A, maximum {pae.max():.1f} A | confidence: pLDDT from the JSON mean {np.mean(sc['plddt']):.1f}" if "plddt" in sc else f"PAE {pae.shape}")
            per_res = pae.mean(1); r = np.corrcoef(per_res, plddt_r)[0, 1]; print(f"correlation between the mean predicted aligned error of a residue and its pLDDT: {r:+.2f} (confident residues should have a low error)")
            check2("the PAE matrix is square and has one row per residue", pae.shape == (Nr, Nr)); check2("residues with high pLDDT have low mean PAE (negative correlation)", r < 0)
            fig, ax = plt.subplots(1, 2, figsize=(10, 3.8)); im = ax[0].imshow(pae, cmap="bwr", vmin=0, vmax=30); plt.colorbar(im, ax=ax[0], fraction=0.046, label="angstrom"); ax[0].set(title="predicted aligned error of your model", xlabel="residue", ylabel="residue")
            ax[1].plot(np.arange(1, Nr + 1), plddt_r, color=COLORS["blue"], label="pLDDT"); ax2 = ax[1].twinx(); ax2.plot(np.arange(1, Nr + 1), per_res, color=COLORS["red"], label="mean PAE"); ax[1].set(xlabel="residue", ylabel="pLDDT"); ax2.set_ylabel("mean PAE (angstrom)"); plt.tight_layout(); plt.show()
    except Exception as e: print("scores could not be analysed:", type(e).__name__, e)
elif pdb_path is not None: print("No scores file found next to the structure: the confidence is read from the B-factor column only.")
if pdb_path is not None: print(f"\nPart II: {sum(ok for _, ok in checks2)} of {len(checks2)} checks passed")

### What to look for
* **Peptide geometry.** Successive C$\alpha$ atoms are about 3.8 \AA{} apart in any model: a sanity check of the file.
* **An exactly three-dimensional map.** The distance map of a model is built from coordinates, so only three eigenvalues are non-zero; a map *predicted* by a network, on the other hand, generally is not, which is why AlphaFold1 had to optimize coordinates against it.
* **Confidence and error.** Residues with a high pLDDT should have a low mean predicted aligned error; low-confidence regions (often the termini or flexible loops) show up in both.
* **Your run against the example.** Compare the contact counts and the detections with the notebook's example; a protein with antiparallel strands has anti-diagonal detections, which the example does not.